# Personalized Hotel Ranking Engine

Four-track ranking project with top-k recommendation inference and monitoring hooks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_expedia_data, prepare_candidate_scoring_table, query_time_split, build_reference_baselines
from src.features import build_preprocessor, prepare_model_inputs
from src.ranking import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    build_topk_recommendations,
    save_inference_bundle,
)
from src.evaluation import build_leaderboard

SEED = 42
PROJECT_NAME = 'personalized-hotel-ranking-engine'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'expedia'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Return top-5 hotel recommendations per search with strong booking relevance.

Primary metric: MAP@5
Secondary: HitRate@5
Tertiary: inference latency

## 2) Dataset Access and Data Dictionary

Dataset: Expedia Hotel Recommendations (`train.csv`).

The project reformulates logs into supervised candidate scoring rows.

In [2]:
data = load_expedia_data(RAW_DIR, sample_frac=0.005, random_state=SEED)
data['train'].shape

(188351, 24)

## 3) Data Cleaning and Leakage Checks

- Construct label from booking signal
- Group-safe negative sampling
- Remove leakage-prone post-event fields

In [3]:
candidate_df = prepare_candidate_scoring_table(data['train'], negative_ratio=5, random_state=SEED)
candidate_df[['label']].value_counts()

label
0        173590
1         14750
Name: count, dtype: int64

## 4) Feature Engineering

- Context and hotel candidate attributes
- Date-derived search signals
- Categorical encoding and numeric scaling

In [4]:
train_df, holdout_df = query_time_split(candidate_df, holdout_frac=0.2, random_state=SEED)
preprocessor = build_preprocessor(train_df.drop(columns=['label', 'srch_id', 'date_time'], errors='ignore'))
X_train, X_holdout, y_train, y_holdout, holdout_meta = prepare_model_inputs(
    train_df=train_df,
    holdout_df=holdout_df,
    preprocessor=preprocessor,
    target_col='label',
    group_col='srch_id',
    item_col='hotel_cluster',
)
X_train.shape, X_holdout.shape

((150672, 19), (37668, 19))

## 5) Validation Strategy

- Query/time-aware holdout split
- Ranking evaluation at search-id level
- Business metrics centered on top-5 usefulness

In [5]:
summary = pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'positive_rate': [train_df['label'].mean(), holdout_df['label'].mean()]
})
summary

,split,rows,positive_rate
0,train,150672,0.080533
1,holdout,37668,0.069449


## 6) Track 1: LazyPredict Discovery Lab

Run candidate scoring discovery on encoded matrix.

In [6]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

,Model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
0,NearestCentroid,0.567500,0.636346,0.636346,0.676680,0.014288
1,QuadraticDiscriminantAnalysis,0.905833,0.507775,0.567788,0.898833,0.029651
2,LabelSpreading,0.874167,0.504362,NaN,0.882398,1.047747
3,ExtraTreeClassifier,0.860000,0.503540,0.503540,0.874828,0.017897
4,LabelPropagation,0.872500,0.503477,NaN,0.881446,0.768277
5,XGBClassifier,0.933333,0.502276,0.562187,0.910590,0.756970
6,AdaBoostClassifier,0.941667,0.500000,0.601410,0.913376,0.175544
7,LogisticRegression,0.941667,0.500000,0.630822,0.913376,0.022364
8,SVC,0.941667,0.500000,0.477990,0.913376,0.249684
9,SGDClassifier,0.941667,0.500000,0.537699,0.913376,0.039879


## 7) Selection of Top 3 Eligible Models

Only top-3 eligible families from LazyPredict proceed to manual lab.

In [7]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table=lazy_table,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    holdout_meta=holdout_meta,
    random_state=SEED,
)
eligible_table, top3_families

(               lazy_model               family  map_at_5  hit_rate_at_5  \
 0  RandomForestClassifier        random_forest  1.005989            1.0   
 1           XGBClassifier              xgboost  1.005989            1.0   
 2      LogisticRegression  logistic_regression  1.005607            1.0   
 
    train_time_sec  infer_latency_ms  p95_latency_ms  eligible eligibility_note  
 0       15.604786         34.340990       36.388262      True         eligible  
 1        7.139588          0.917196        3.328655      True         eligible  
 2        0.337299          0.042956        0.051078      True         eligible  ,
 ['logistic_regression', 'random_forest', 'xgboost'])

## 8) Track 2: Manual Engineering Lab

Manual top-3 candidate scoring with explicit score-to-top-k conversion.

In [8]:
manual_results, manual_models, manual_scored = run_manual_engineering_track(
    top3_families=top3_families,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    holdout_meta=holdout_meta,
    random_state=SEED,
)
manual_results[['model_name', 'map_at_5', 'hit_rate_at_5', 'p95_latency_ms']]

,model_name,map_at_5,hit_rate_at_5,p95_latency_ms
0,random_forest,1.005989,1.0,36.271510
1,xgboost,1.005989,1.0,7.145054
2,logistic_regression,1.005607,1.0,0.053837


## 9) Track 3: FLAML Optimization Lab

Budget-aware challenger search with ranking-centric evaluation.

In [9]:
flaml_result = run_flaml_track(
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    holdout_meta=holdout_meta,
    time_budget=120,
    random_state=SEED,
)
flaml_result

[flaml.automl.logger: 06-11 19:53:23] {2375} INFO - task = classification


[flaml.automl.logger: 06-11 19:53:23] {2386} INFO - Evaluation method: cv


[flaml.automl.logger: 06-11 19:53:23] {2489} INFO - Minimizing error metric: 1-ap


[flaml.automl.logger: 06-11 19:53:23] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost', 'rf', 'extra_tree', 'lrl1']


[flaml.automl.logger: 06-11 19:53:23] {2911} INFO - iteration 0, current learner lgbm


...

[flaml.automl.logger: 06-11 19:55:26] {3359} INFO - retrain lgbm for 1.3s


[flaml.automl.logger: 06-11 19:55:26] {3362} INFO - retrained model: LGBMClassifier(colsample_bytree=0.5159630257937373,
               learning_rate=0.0811002415722797, max_bin=511,
               min_child_samples=87, n_estimators=377, n_jobs=-1, num_leaves=8,
               reg_alpha=0.4898000784473407, reg_lambda=0.07067449013601991,
               verbose=-1)


[flaml.automl.logger: 06-11 19:55:26] {2636} INFO - fit succeeded


[flaml.automl.logger: 06-11 19:55:26] {2637} INFO - Time taken to find the best model: 117.15281891822815


{'model_name': 'lgbm',
 'library_source': 'flaml',
 'map_at_5': 1.0057976554536188,
 'hit_rate_at_5': 1.0,
 'train_time_sec': 122.84382559498772,
 'infer_latency_ms': 0.5921664108367016,
 'p95_latency_ms': 2.0970499055692935,
 'interpretability_note': 'FLAML challenger for ranking candidate scoring',
 'best_config': {'n_estimators': 377,
  'num_leaves': 8,
  'min_child_samples': 87,
  'learning_rate': 0.0811002415722797,
  'log_max_bin': 9,
  'colsample_bytree': 0.5159630257937373,
  'reg_alpha': 0.4898000784473407,
  'reg_lambda': 0.07067449013601991},
 'best_loss': 0.856079513106469,
 'scored':        srch_id  hotel_cluster  label     score
 0        30268             85      0  0.120422
 1       127890              5      0  0.047274
 2        54655             42      1  0.048073
 3       171449             16      0  0.050985
 4         6478             77      0  0.073016
 ...        ...            ...    ...       ...
 37663    84648             29      0  0.069792
 37664   1160

## 10) Track 4: PyCaret Experiment Lab

PyCaret candidate-scoring experiment evaluated with ranking metrics.

In [10]:
pycaret_train = train_df.drop(columns=['date_time'], errors='ignore').copy()
pycaret_holdout = holdout_df.drop(columns=['date_time'], errors='ignore').copy()
pycaret_result = run_pycaret_track(
    train_df=pycaret_train,
    holdout_df=pycaret_holdout,
    target_col='label',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_expedia_ranking',
)
pycaret_result

{'model_name': 'pycaret_failed',
 'library_source': 'pycaret',
 'map_at_5': nan,
 'hit_rate_at_5': nan,
 'train_time_sec': nan,
 'infer_latency_ms': nan,
 'p95_latency_ms': nan,
 'interpretability_note': "PyCaret unavailable or failed: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=10, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')",
 'scored': Empty DataFrame
 Columns: []
 Index: [],
 'status': 'failed'}

## 11) Unified Leaderboard and Final Model Ranking

Includes popularity/recent-booking baselines plus all 4 tracks.

In [11]:
baseline_rows, holdout_with_baselines = build_reference_baselines(train_df, holdout_df)
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    baseline_rows=baseline_rows,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
)
leaderboard.head(10)

,project_name,task_type,library_source,model_name,cv_metric_mean,cv_metric_std,holdout_primary_metric,holdout_secondary_metric,holdout_tertiary_metric,calibration_metric,train_time_sec,infer_latency_ms,p95_latency_ms,model_size_mb,retrain_time_sec,interpretability_note,rank_score,final_rank
0,personalized-hotel-ranking-engine,ranking_candidate_scoring,lazypredict,logistic_regression,NaN,NaN,1.005607,1.0,0.051078,NaN,0.337299,0.042956,0.051078,NaN,0.337299,eligible,0.953742,1
1,personalized-hotel-ranking-engine,ranking_candidate_scoring,manual,logistic_regression,NaN,NaN,1.005607,1.0,0.053837,NaN,0.314440,0.046189,0.053837,NaN,0.314440,Manual candidate-scoring model: logistic_regre...,0.953738,2
2,personalized-hotel-ranking-engine,ranking_candidate_scoring,flaml,lgbm,NaN,NaN,1.005798,1.0,2.097050,NaN,122.843826,0.592166,2.097050,NaN,122.843826,FLAML challenger for ranking candidate scoring,0.951061,3
3,personalized-hotel-ranking-engine,ranking_candidate_scoring,lazypredict,xgboost,NaN,NaN,1.005989,1.0,3.328655,NaN,7.139588,0.917196,3.328655,NaN,7.139588,eligible,0.949499,4
4,personalized-hotel-ranking-engine,ranking_candidate_scoring,baseline,popularity_baseline,NaN,NaN,1.005161,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Global popularity ranker,0.948935,5
5,personalized-hotel-ranking-engine,ranking_candidate_scoring,baseline,recent_booking_baseline,NaN,NaN,1.005161,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Recent booking popularity ranker,0.948935,6
6,personalized-hotel-ranking-engine,ranking_candidate_scoring,manual,xgboost,NaN,NaN,1.005989,1.0,7.145054,NaN,4.236566,1.059862,7.145054,NaN,4.236566,Manual candidate-scoring model: xgboost,0.944255,7
7,personalized-hotel-ranking-engine,ranking_candidate_scoring,manual,random_forest,NaN,NaN,1.005989,1.0,36.271510,NaN,15.762972,34.557751,36.271510,NaN,15.762972,Manual candidate-scoring model: random_forest,0.904233,8
8,personalized-hotel-ranking-engine,ranking_candidate_scoring,lazypredict,random_forest,NaN,NaN,1.005989,1.0,36.388262,NaN,15.604786,34.340990,36.388262,NaN,15.604786,eligible,0.904072,9
9,personalized-hotel-ranking-engine,ranking_candidate_scoring,pycaret,pycaret_failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PyCaret unavailable or failed: ('Pycaret only ...,-0.004574,10


In [12]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_expedia_ranking.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'holdout_secondary_metric', 'final_rank']].head(10)

,model_name,library_source,holdout_primary_metric,holdout_secondary_metric,final_rank
0,logistic_regression,lazypredict,1.005607,1.0,1
1,logistic_regression,manual,1.005607,1.0,2
2,lgbm,flaml,1.005798,1.0,3
3,xgboost,lazypredict,1.005989,1.0,4
4,popularity_baseline,baseline,1.005161,1.0,5
5,recent_booking_baseline,baseline,1.005161,1.0,6
6,xgboost,manual,1.005989,1.0,7
7,random_forest,manual,1.005989,1.0,8
8,random_forest,lazypredict,1.005989,1.0,9
9,pycaret_failed,pycaret,NaN,NaN,10


## 12) Business Recommendation

After run completion, explain winner, safer second-best scenario, and excluded tradeoff.

## 13) Inference / Deployment Path

Generate top-5 recommendations from winning scorer and save sample output.

In [13]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_scored:
    scored_df = manual_scored[winner['model_name']]
    if winner['model_name'] in manual_models:
        save_inference_bundle(manual_models[winner['model_name']], preprocessor, ARTIFACT_DIR)
elif winner['library_source'] == 'flaml':
    scored_df = flaml_result.get('scored', pd.DataFrame())
elif winner['library_source'] == 'pycaret':
    scored_df = pycaret_result.get('scored', pd.DataFrame())
elif winner['model_name'] == 'popularity_baseline':
    scored_df = holdout_with_baselines[['srch_id', 'hotel_cluster', 'label', 'score_popularity']].rename(columns={'score_popularity': 'score'})
else:
    scored_df = holdout_with_baselines[['srch_id', 'hotel_cluster', 'label', 'score_recent_booking']].rename(columns={'score_recent_booking': 'score'})

top5 = build_topk_recommendations(scored_df, k=5)
top5.to_csv(ARTIFACT_DIR / 'top5_sample_recommendations.csv', index=False)
top5.head()

,srch_id,top_k_items,top_k_scores
0,0,[66],[0.020833333333333332]
1,2,[51],[0.09219858156028368]
2,4,[68],[0.10119047619047619]
3,6,[7],[0.1875]
4,7,[56],[0.08900523560209424]


## 14) Monitoring / Drift / Retraining Plan

Track MAP@5 and HitRate@5 over sliding windows, candidate coverage drift, and cold-start performance by segment.

## 15) Limitations and Next Steps

- Add richer user history features
- Add explicit diversity constraints in top-k output
- Add counterfactual policy evaluation before full rollout